# [RAGAs를 이용한 데이터셋 생성](https://docs.ragas.io/en/stable/howtos/customizations/testgenerator/_language_adaptation/#generate)

## Load Dataset

In [ ]:
import pandas as pd 

df = pd.read_csv("./data/cleaning/nutrition.csv")

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")
df.head(2)

## Loader

In [ ]:
from langchain_community.document_loaders import CSVLoader

# nutrition.csv -> caption만 로드 진행할 수 있는 객체
loader = CSVLoader(
    file_path="./data/cleaning/nutrition.csv" # 파일명 
    , source_column='video_url' # 소스 컬럼 
    , content_columns=['title_caption'] # 컨텐츠(학습) 컬럼 
    , encoding="utf-8"
)

In [ ]:
docs = loader.load() # load()를 이용해서 데이터 로드 진행..

In [ ]:
# df.shape -> (데이터의 수(=유튜브 영상의 수), 컬럼의 수)
len(docs), df.shape

In [ ]:
docs[0].metadata['source']

In [ ]:
docs[0].page_content

## Model

### [OpenAI API Key](https://platform.openai.com/settings/organization/api-keys)

In [ ]:
from dotenv import load_dotenv 

# .env 파일에 있는 데이터를 환경변수에 등록해주는 함수 
load_dotenv()

## 학습 데이터셋

### [문장을 생성하는 LLM](https://platform.openai.com/docs/models)

In [ ]:
# Langchain LLM 객체를 ragas가 인식할 수 있게 변형
# Wrapper -> 감싸다. 또는 형변환 된다...
from ragas.llms import LangchainLLMWrapper
# Langchain에서 인식할 수 있는 OpenaAI LLM 객체
from langchain_openai import ChatOpenAI

# 문장을 생성하는 LLM -> 질문 & 답변 생성을 목적으로...
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini"
))

### [Embedding model](https://platform.openai.com/docs/guides/embeddings/embedding-models#embedding-models)

In [ ]:
# RAGAs의 최신 OpenAIEmbeddings 사용 (client 필요)
from ragas.embeddings import OpenAIEmbeddings
from openai import OpenAI

# Embedding model -> 문장을 벡터로 변환하는 모델...
generator_embeddings = OpenAIEmbeddings(
    # OpenAI client 생성 (환경변수의 OPENAI_API_KEY를 자동으로 사용)
    client=OpenAI(),
    model="text-embedding-3-small"
)

### 페르소나 정의(중요)
> 어떤 질문/답변 쌍을 만들지 정의함 

In [ ]:
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="a pediatric nutrition expert", # 소아과 의사 선생님
        # 일반 대중이 이해하기 쉽게 영양을 설명해주는 의사
        role_description=""""
        You are a pediatric nutrition expert who explains infant and toddler nutrition clearly, accurately, and in an easy-to-understand way for parents and caregivers.
        """,
    ),
    Persona(
        name="a child nutrition specialist", # 소아과 의사 선생님
        # 일반 대중이 이해하기 쉽게 영양을 설명해주는 의사
        role_description=""""
        You are a child nutrition specialist who helps parents understand what and how much infants and toddlers should eat, providing practical, age-appropriate feeding guidance.
        """,
    ),
    Persona(
        name="a pediatrician", # 소아과 의사 선생님
        # 일반 대중이 이해하기 쉽게 영양을 설명해주는 의사
        role_description=""""
        You are a pediatrician with expertise in child nutrition, explaining infant and toddler dietary needs, nutrients, and feeding concerns using evidence-based guidance.
        """,
    ),
]

### 학습용 데이터를 만드는 객체

In [ ]:
from ragas.testset import TestsetGenerator

# 학습용 데이터를 만드는 객체
generator = TestsetGenerator(
    llm=generator_llm, embedding_model=generator_embeddings, 
    persona_list=personas
)

NERExtractor: 
> RAG 평가용 테스트셋 생성 시, 문서에 등장하는 핵심 엔티티를 뽑아 질문을 자동 생성하는 클래스입니다.

In [ ]:
from ragas.testset.transforms.extractors.llm_based import NERExtractor

transforms = [NERExtractor(llm=generator_llm)]

SingleHopSpecificQuerySynthesizer
> Ragas의 테스트셋 생성 기능에서, 단일 문서 기반(single-hop)으로 “특정하고 구체적인 질문”을 자동 생성하는 Synthesizer 클래스입니다.

In [ ]:
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer
)

distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
]

for query, _ in distribution:
    prompts = await query.adapt_prompts("korean", llm=generator_llm)
    query.set_prompts(**prompts)

### 학습용 데이터셋 생성

In [ ]:
dataset = generator.generate_with_langchain_docs(
    docs[:],
    testset_size=200, # 총 몇개의 질문/답변 쌍을 만들지
    transforms=transforms,
    query_distribution=distribution,
)

## 생성된 데이터셋 확인

In [ ]:
eval_dataset = dataset.to_evaluation_dataset()

In [ ]:
print("Query:", eval_dataset[0].user_input)
print("Reference:", eval_dataset[0].reference)

In [ ]:
# 생성된 테스트셋을 pandas DataFrame으로 변환
eval_df = eval_dataset.to_pandas()
eval_df.shape

In [ ]:
eval_df.head(2)

### reference_video_url 추가하기 

In [ ]:
eval_df.loc[0,['reference_contexts']].values[0][0].replace("title_caption: ", "")

In [ ]:
# df[['video_url', 'title_caption']].head(2)
df.loc[0,['title_caption']].values[0]

In [ ]:
eval_df['reference_title_caption'] = eval_df['reference_contexts'].map(
    lambda x: x[0].replace("title_caption: ", "").strip()
)

eval_df[['reference_title_caption', 'reference_contexts']].head(2)

In [ ]:
eval_df = eval_df.merge(
    right=df[['video_url', 'title_caption']], how='inner',
    left_on="reference_title_caption", right_on="title_caption"
)

In [ ]:
eval_df.rename(
    columns={
        "video_url": "reference_video_url"
    }, inplace= True
)

In [ ]:
eval_df[['user_input', 'reference', 'reference_video_url']].head()

In [ ]:
eval_df[['user_input', 'reference', 'reference_video_url']].tail()

> 중복 데이터 제거 

In [ ]:
eval_df = eval_df[['user_input', 'reference', 'reference_video_url']]
print(f"befer: {eval_df.shape}")

eval_df.drop_duplicates(inplace=True, subset=['user_input', 'reference'])
print(f"after: {eval_df.shape}")

> 결측치 확인

In [ ]:
eval_df.isnull().sum()

### 저장하기 

> 데이터를 랜덤하게 정렬하기 

- `sample(frac=1)` : 전체 행을 랜덤 셔플
- `random_state=42` : 재현 가능하도록 시드 고정 (선택)
- `reset_index(drop=True)` : 기존 index 제거 후 0부터 새로 생성

In [ ]:
save_df = eval_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
save_df.head()

> 저장하기 

In [ ]:
save_df.to_csv("./data/training/nutrition.csv", index=False, header=True)